# Chains in LangChain

## Outline

* LLMChain 过时了，用 LCEL (LangChain Expression Language) 的管道符 | 来实现
* Sequential Chains 用于将复杂的任务分解为 A -> B -> C 这样顺序执行的子任务。例如：Chain A (提炼用户意图) -> Chain B (执行搜索) -> Chain C (总结结果)。也过时了
  * SimpleSequentialChain
  * SequentialChain
* Router Chain 让 LLM 作为“大脑”，先决定用户的查询应该交给哪个子 Chain 或工具来处理。这是一种初级的 Agent 实现形式， 也过时了

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

Note: LLM's do not always produce the same results. When executing the code in your notebook, you may get slightly different answers that those in the video.


注意：LLM 算法并非总能产生相同的结果。在笔记本中运行代码时，您可能会得到与视频中略有不同的结果。

本代码用千问模型，下面代码不需要执行

In [ ]:
# account for deprecation of LLM model
import datetime
# Get the current date
current_date = datetime.datetime.now().date()

# Define the date after which the model should be set to "gpt-3.5-turbo"
target_date = datetime.date(2024, 6, 12)

# Set the model variable based on the current date
if current_date > target_date:
    llm_model = "gpt-3.5-turbo"
else:
    llm_model = "gpt-3.5-turbo-0301"

In [3]:
import pandas as pd
df = pd.read_csv('Data.csv')

In [4]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


In [6]:
# 定义大模型，后面都要用到
# 用opanai兼容方式定义通义千问模型
from langchain_openai import ChatOpenAI

llm_model = "qwen-max"
llm = ChatOpenAI(
    temperature=0.9, 
    model=llm_model,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY")
    )

In [ ]:
# 导入通义千问模型
from langchain_community.chat_models.tongyi import ChatTongyi
# 初始化模型,由于会下载分词配置会失败，所以需要用专门的ChatTongyi，不然会调用openai的会报错
tongyiLLM = ChatTongyi(
    model="qwen-turbo",  # 或其他通义千问模型
    dashscope_api_key=os.getenv("DASHSCOPE_API_KEY"),  # 通义千问 API key
    temperature=0.9, 
)

## LLMChain（基础链）
### 老方法
以下是老方法的实现代码，已经过时

In [14]:
from langchain_classic.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain

# 定义 Prompt Template，包含一个变量 {product}
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

In [43]:
# 实例化 LLMChain：将 llm 和 prompt 组合起来
chain = LLMChain(llm=llm, prompt=prompt)

In [ ]:
# 运行 Chain，传入变量 product
product = "Queen Size Sheet Set"
chain.run(product)
# LLMChain 接收 product -> 格式化 Prompt -> 调用 LLM -> 返回结果

C:\Users\bangsun\AppData\Local\Temp\ipykernel_5764\550859008.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  chain.run(product)


"Choosing a name for a company that specializes in Queen Size Sheet Sets can be both creative and strategic. The name should ideally convey comfort, luxury, and quality, while also being memorable and easy to pronounce. Here are some suggestions:\n\n1. **Royal Comfort Linens**\n2. **Queenly Sheets Co.**\n3. **Regal Bedding**\n4. **Majestic Sleep**\n5. **Royal Weave**\n6. **Crown Comforts**\n7. **Queens Haven Linens**\n8. **Elegant Slumber**\n9. **Sovereign Sheets**\n10. **Luxury Queen Linens**\n11. **Royal Rest**\n12. **Imperial Sheets**\n13. **Queen's Choice Linens**\n14. **Noble Nights**\n15. **Grandeur Bedding**\n\nEach of these names aims to evoke a sense of luxury and quality, which is often associated with queen-sized bedding. You can choose the one that best aligns with your brand's identity and target market."

### 1.0版本方法
以下是1.0版本提倡的方法

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 初始化大模型组件，就是最开始定义的llm,这里不重复定义了
llm = llm

# 2. 定义 Prompt Template
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)

# 3. 使用 LCEL 管道符 | 链接（LLMChain 替代）
# LangChain Expression Language (LCEL) 的典型用法，它使用管道操作符 | 将多个组件串联成一个执行链。
# 这种设计灵感来源于Unix管道系统，让数据像流水线一样从左向右流动，每个环节只负责单一任务。
# LangChain重写了python的管道操作符，使其能够处理复杂的LLM工作流。
# 这行代码具体做了以下事情
# - prompt：定义了一个 ChatPromptTemplate，用于生成提示语。 
# - llm：表示一个语言模型组件，负责处理提示语并生成响应。   将 prompt 的输出（HumanMessage）自动作为输入  
# - StrOutputParser()：定义了一个输出解析器，用于将语言模型的输出转换为字符串格式。 
# 核心：所有langchain的类都继承了Runnable接口，所以都可以用管道符连接且都有invoke、stream、batch方法，
# 用管道符链接其实就是从左往右依次执行对应类的invoke或stream或batch方法，前一个的输出作为后一个的输入。
# 最终，lcel_chain 代表了一个完整的处理链，可以直接调用 invoke、stream、batch 方法来执行整个流程。
lcel_chain = prompt | llm | StrOutputParser()
# 上面代码等价于下面代码，返回的是一个 RunnableSequence 对象
# lcel_chain = RunnableSequence(
#     first=prompt,
#     middle=[llm],  # 中间组件列表
#     last=StrOutputParser()
# )
# 检查类型
print("=== LCEL Chain 信息 ===")
print(f"类型：{type(lcel_chain)}")   # 确认对象类型
# print(f"文档：{help(lcel_chain)}")   # 查看文档
# 查看内部结构
# 需要安装包pip install grandalf
print("=== LCEL Chain 结构图 ===")
print(lcel_chain.get_graph().draw_ascii())

# 4. 调用
product = "Queen Size Sheet Set"  # 女王尺寸床单套件
print("=== 调用 LCEL Chain ===")
# 如果这里是lcel_chain.stream()，则是依次调用链路上类的stream方法
response = lcel_chain.invoke({"product": product})
print(response)

=== LCEL Chain 信息 ===
类型：<class 'langchain_core.runnables.base.RunnableSequence'>
=== LCEL Chain 结构图 ===
     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
=== 调用 LCEL Chain ===
Choosing a name for a company that specializes in Queen Size Sheet Sets can be both creative and strategic. The name 

## SimpleSequentialChain（简单顺序链）
用于将两个或多个 Chain 按顺序连接，特点是前一个 Chain 的完整输出作为后一个 Chain 的唯一输入。

### 老方法
以下是老方法的实现代码，已经过时

In [15]:
from langchain_classic.chains import SimpleSequentialChain

In [16]:
# llm = ChatOpenAI(temperature=0.9, model=llm_model)

# Chain 1: 命名公司
# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

C:\Users\bangsun\AppData\Local\Temp\ipykernel_36236\2351075888.py:11: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain_one = LLMChain(llm=llm, prompt=first_prompt)


In [17]:
# Chain 2: 编写描述 (接收 Chain 1 的输出作为 {company_name})
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words description for the following \
    company:{company_name}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [18]:
# 组合简单顺序链
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [19]:
# 运行 Chain
# 输入 {product} -> Chain 1 输出 {company_name} -> Chain 2 输出最终描述
product = "Queen Size Sheet Set" 
overall_simple_chain.run(product)

C:\Users\bangsun\AppData\Local\Temp\ipykernel_36236\1923811914.py:4: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  overall_simple_chain.run(product)




> Entering new SimpleSequentialChain chain...
When naming a company that specializes in queen size sheet sets, you want a name that is both memorable and descriptive. Here are some suggestions:

1. **Queenly Comfort**
2. **Royal Sheets Co.**
3. **Regal Bedding**
4. **QueenSize Linens**
5. **Majestic Sheets**
6. **Crown Comforts**
7. **Queen’s Quarters**
8. **Royal Rest**
9. **Luxury Queen Linens**
10. **Elegant Queen Sheets**

Each of these names conveys a sense of quality, comfort, and luxury, which are key attributes for a company that provides high-quality bedding. Choose one that best aligns with your brand's identity and the experience you want to provide to your customers.
Elegant Queen Sheets offers luxurious, high-quality bedding designed for ultimate comfort and a regal sleeping experience.

> Finished chain.


'Elegant Queen Sheets offers luxurious, high-quality bedding designed for ultimate comfort and a regal sleeping experience.'

### 1.0版本方法
以下是1.0版本提倡的方法

In [32]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. 初始化大模型组件，就是最开始定义的llm,这里不重复定义了
llm = llm
parser = StrOutputParser()

# Chain 1: 命名公司
chain_one = (
    ChatPromptTemplate.from_template("What is the best name to describe a company that makes {product}?")
    | llm
    | parser
)

# Chain 2: 编写描述 (接收 Chain 1 的输出作为 {company_name})
chain_two = (
    ChatPromptTemplate.from_template("Write a 20 words description for the following company:{company_name}")
    | llm
    | parser
)

# 组合：将 Chain 1 的输出（默认作为唯一的下一个输入）直接管道给 Chain 2
lcel_simple_chain = chain_one | chain_two
print("=== LCEL Chain 结构图 ===")
print(lcel_simple_chain.get_graph().draw_ascii())
# 调用
product = "Queen Size Sheet Set"
response = lcel_simple_chain.invoke({"product": product})
print("=== LCEL Chain 结果 ===")
print(response)

=== LCEL Chain 结构图 ===
     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *            

## SequentialChain（复杂顺序链）

比 SimpleSequentialChain 更灵活，支持多个输入和多个输出，需要通过 output_key 和 input_variables 显式管理数据流。

### 老方法
以下是老方法的实现代码，已经过时

In [20]:
from langchain_classic.chains import SequentialChain

In [21]:
# llm = ChatOpenAI(temperature=0.9, model=llm_model)

# Chain 1: 翻译评论。输入: Review, 输出: English_Review
# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:"
    "\n\n{Review}"
)
# chain 1: input= Review and output= English_Review
chain_one = LLMChain(llm=llm, prompt=first_prompt, 
                     output_key="English_Review"
                    )


In [22]:
# Chain 2: 总结英文评论。输入: English_Review, 输出: summary
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:"
    "\n\n{English_Review}"
)
# chain 2: input= English_Review and output= summary
chain_two = LLMChain(llm=llm, prompt=second_prompt, 
                     output_key="summary"
                    )


In [23]:
# Chain 3: 判断原始语言。输入: Review, 输出: language (与 Chain 2 并行使用了 Review)
# prompt template 3: translate to english
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="language"
                      )


In [24]:
# Chain 4: 编写后续回复。输入: summary, language, 输出: followup_message
# prompt template 4: follow up message
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
# chain 4: input= summary, language and output= followup_message
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )


In [25]:
# 组合 SequentialChain
# overall_chain: input= Review 
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four], # 按顺序执行的子链
    input_variables=["Review"], # 整个链的初始输入变量
    output_variables=["English_Review", "summary", "followup_message"], # 整个链的最终输出变量
    verbose=True
)

In [26]:
# 运行 Chain，输入原始评论
review = df.Review[5] 
overall_chain(review) 
# SequentialChain 追踪 Review -> English_Review, Review -> language, 
# (English_Review) -> summary, (summary, language) -> followup_message

C:\Users\bangsun\AppData\Local\Temp\ipykernel_36236\3298087519.py:3: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  overall_chain(review)




> Entering new SequentialChain chain...

> Finished chain.


{'Review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?",
 'English_Review': "I find the taste mediocre. The foam doesn't last, which is strange. I buy the same ones in stores and the taste is much better...\nOld batch or counterfeit!?",
 'summary': "The reviewer finds the product's taste mediocre and its foam short-lived, questioning whether it might be an old batch or counterfeit compared to better-tasting store-bought versions.",
 'followup_message': "Cher client,\n\nNous sommes sincèrement désolés d'apprendre votre insatisfaction concernant notre produit. Nous prenons très au sérieux vos remarques sur la qualité du goût et la durée de vie de la mousse. Votre expérience n'est pas celle que nous souhaitons offrir, et nous comprenons votre inquiétude quant à la possibilité qu'il s'agisse d'un lot ancien ou d'un produit contrefait.\n\nNous aimerions vous assurer que l

### 1.0版本方法
以下是1.0版本提倡的方法

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda

# 1. 初始化大模型组件，就是最开始定义的llm,这里不重复定义了
llm = llm
parser = StrOutputParser()
review = df.Review[5] # 输入

# 1. 定义子 Chain
# Chain 1: 翻译 Review -> English_Review (串行)
chain_one = (
    ChatPromptTemplate.from_template("Translate the following review to english:\n\n{Review}")
    | llm
    | parser
).with_config(run_name="Translate") # 添加 run_name 方便调试

# Chain 3: 判断 Review -> language (与 Chain 1 并行执行)
chain_three = (
    ChatPromptTemplate.from_template("What language is the following review:\n\n{Review}")
    | llm
    | parser
).with_config(run_name="DetectLanguage")

# 2. 定义整体复合链
lcel_complex_chain = (
    # 步骤 A: 并行执行 Chain 1 和 Chain 3。
    # RunnableParallel 允许您指定哪些键从上游（即初始输入 Review）流向哪个子 Chain
    RunnableParallel(
        # 串行部分：翻译 Review
        English_Review=chain_one,  # 直接接收完整输
        # 并行部分：判断语言
        language=chain_three,
        Review=RunnablePassthrough() # 保留原始Review
    )
    # 步骤 B: 基于翻译结果生成摘要，它依赖于 English_Review
    .assign(
        # 自动接收当前状态字典
        summary=ChatPromptTemplate.from_template("Can you summarize the following review in 1 sentence:\n\n{English_Review}")
        | llm
        | parser
    )
    # 步骤 C: 生成本地化跟进消息，它依赖于 summary 和 language
    .assign(
        followup_message=ChatPromptTemplate.from_template(
            "Write a follow up response to the following \"\n    \"summary in the specified language:\"\n    \"\\n\\nSummary: {summary}\\n\\nLanguage: {language}\""
        )
        | llm
        | parser
    )
    # 步骤 D: 使用列表传递键名
    .pick(["English_Review", "summary", "followup_message"])
)

print("=== LCEL Chain 结构图 ===")
print(lcel_complex_chain.get_graph().draw_ascii())
# 调用
response = lcel_complex_chain.invoke({"Review": review})
print("=== LCEL Chain 结果 ===")
print(response)

=== LCEL Chain 结构图 ===
                      +-----------------------------------------------+                     
                      | Parallel<English_Review,language,Review>Input |                     
                      +-----------------------------------------------+                     
                          *******              *              *******                       
                     *****                     *                     *****                  
                 ****                          *                          *******           
+--------------------+              +--------------------+                       ***        
| ChatPromptTemplate |              | ChatPromptTemplate |                         *        
+--------------------+              +--------------------+                         *        
           *                                   *                                   *        
           *                                   

## Router Chain
用于根据用户输入，由一个 路由 LLM 决定将请求导向哪个目标 Chain。

### 老方法
以下是老方法的实现代码，已经过时

In [35]:
# 翻译如下：
# 你是一位非常聪明的物理学教授。
# 您擅长以简洁易懂的方式回答有关物理的问题。
# 当你不知道一个问题的答案时，你承认你不知道
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""

# 翻译如下：
# 你是一位非常优秀的数学家。
# 你很擅长回答数学问题。
# 你之所以如此优秀，是因为你能将难题分解成各个组成部分，
# 解答每个组成部分的问题，然后将它们组合起来回答更广泛的问题。
math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""


# 翻译如下：
# 你是一位非常优秀的史学家。
# 你对人有着深刻的了解和理解。
# 涵盖多个历史时期的事件和背景。
# 你具备思考、反思、辩论、讨论和评价过去的能力。你尊重历史证据，并能够运用历史证据来支持你的解释和判断
history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


# 翻译如下：
# 你是一位成功的计算机科学家。
# 你充满热情，富有创造力，善于协作，思维前瞻，自信满满，拥有强大的问题解决能力，
# 精通理论和算法，并且沟通能力出色。你非常擅长解答编程问题。
# 你之所以如此优秀，是因为你懂得如何用机器易于理解的指令步骤来描述解决方案，
# 并且能够选择时间复杂度和空间复杂度之间达到良好平衡的方案。
computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

In [36]:
# 1. 定义多个专家 Prompt (物理、数学、历史、计算机科学)
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    }
]

In [ ]:
# MULTI_PROMPT_ROUTER_TEMPLATE 是一个复杂的 Prompt，要求 LLM 输出 JSON 格式的决策
# 决定 destination 和 next_inputs #
# 翻译如下
# 给定一段原始文本输入到语言模型中，请选择最适合该输入的模型提示。
# 系统会提供可用提示的名称以及每个提示的适用场景描述。
# 如果您认为修改原始输入最终能使语言模型给出更好的响应，也可以进行修改。

# 请记住：“next_inputs”可以只是原始输入，如果您认为不需要任何修改。
# 请记住：“destination”必须是下面指定的候选提示名称之一，或者，
# 如果输入不适合任何候选提示，则可以是“DEFAULT”。
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [29]:
from langchain_classic.chains.router import MultiPromptChain
from langchain_classic.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain_classic.prompts import PromptTemplate

In [ ]:
# 2. 创建目标 Chains (Destination Chains)
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
# 3. 定义路由 LLM 的 Prompt (包含所有目标及其描述)
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [37]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [ ]:
# 4. 创建 LLMRouterChain
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),# 解析 LLM 输出的 JSON，获取目的地
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [ ]:
# 5. 组合 MultiPromptChain (Router Chain)
chain = MultiPromptChain(router_chain=router_chain, 
                        destination_chains=destination_chains, # 所有目标 Chain
                        default_chain=default_chain, # 默认 Chain (如果无法路由)
                        )

C:\Users\bangsun\AppData\Local\Temp\ipykernel_5764\3038952769.py:1: LangChainDeprecationWarning: Please see migration guide here for recommended implementation: https://python.langchain.com/docs/versions/migrating_chains/multi_prompt_chain/
  chain = MultiPromptChain(router_chain=router_chain,


In [ ]:
# 运行 Chain：Router LLM 会先判断 "What is black body radiation?" 应该路由给 physics Chain
chain.run("What is black body radiation?")



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}
> Finished chain.


'Black body radiation refers to the electromagnetic radiation emitted by an idealized physical body known as a "black body." A black body is a theoretical object that absorbs all incident electromagnetic radiation, regardless of frequency or angle of incidence. Because it absorbs all incoming light, it appears perfectly black when it\'s cold and not emitting much radiation.\n\nWhen a black body is heated, it emits radiation at various wavelengths, and the spectrum of this radiation depends only on the temperature of the body. The intensity of the radiation peaks at a specific wavelength, which shifts to shorter wavelengths (towards the blue end of the spectrum) as the temperature increases. This relationship is described by Planck\'s law, which was formulated by Max Planck in 1900 and is a fundamental part of quantum mechanics.\n\nIn summary, black body radiation is the characteristic way that an idealized, perfectly absorbing and emitting object radiates energy based on its temperatur

In [36]:
chain.run("what is 2 + 2")



> Entering new MultiPromptChain chain...
math: {'input': 'what is 2 + 2'}
> Finished chain.


'The question "What is 2 + 2?" is a straightforward arithmetic problem.\n\nTo break it down:\n- The number 2 represents two units.\n- The plus sign (+) indicates that we need to add the numbers together.\n- So, 2 + 2 means we are combining two units with another two units.\n\nWhen we add them together, we get:\n\\[ 2 + 2 = 4 \\]\n\nSo, the answer is 4.'

In [37]:
chain.run("Why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
None: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


'Every cell in our body contains DNA because DNA (deoxyribonucleic acid) is the genetic material that carries the instructions for the development, function, and reproduction of all living organisms. Here are a few key reasons why every cell needs DNA:\n\n1. **Genetic Blueprint**: DNA contains the genetic code, which is the blueprint for building and maintaining an organism. This code is essential for the cell to know how to function and what role it should play in the body.\n\n2. **Protein Synthesis**: DNA is transcribed into RNA, which is then translated into proteins. Proteins are crucial for almost all cellular functions, including structural support, enzymatic reactions, signaling, and more. Without DNA, cells would not be able to produce the necessary proteins to carry out their specific roles.\n\n3. **Cell Division and Reproduction**: When cells divide, they need to pass on their genetic information to the new cells. DNA replication ensures that each new cell receives a complete

### 1.0版本方法
以下是1.0版本提倡的方法

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda
from pydantic import BaseModel, Field 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough # 确保导入了 RunnablePassthrough

# 初始化大模型组件，就是最开始定义的llm,这里不重复定义了
llm = llm
parser = StrOutputParser()

# --- 1. 定义路由决策的 Pydantic Schema ---
class Route(BaseModel):
    """用于路由输入到正确目的地的决策"""
    destination: str = Field(description="The name of the prompt template to use ('physics', 'math', 'History', 'computer science', or 'DEFAULT')")
    next_inputs: str = Field(description="A potentially modified version of the original input.")

# --- 2. 定义目标 Chains ---
def create_expert_chain(template):
    # 作用：构建专家 Chain (Prompt | LLM | Parser)
    return ChatPromptTemplate.from_template(template) | llm | parser

# --- 3. 创建路由决策 Chain (LLM 结构化输出) ---
# 定义路由 LLM 的 Prompt (包含所有目标及其描述)
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)

# 核心步骤：强制 LLM 输出 Route Pydantic 结构
# llm.with_structured_output(Route) 作用：通过调用工具或 JSON 模式，强制 LLM 的输出遵循 Route 类的结构。
routing_chain = ChatPromptTemplate.from_template(router_template) | llm.with_structured_output(Route)
# 作用：routing_chain 的最终输出是一个 Pydantic Route 实例 (例如 Route(destination='physics', next_inputs='...'))

# --- 4. 组合 RunnableBranch ---
# 创建条件分支列表 (condition, runnable_to_execute)
conditional_branches = []
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    # 构建专家 Chain，并用 run_name 命名
    chain = create_expert_chain(prompt_template).with_config(run_name=name)
    # 构造 lambda 条件函数
    condition = lambda x, name=name: x.destination == name
    # 构造执行的 Runnable，RunnableLambda是为了将router返回的Route对象里的原始问题next_inputs取出来，作为专家chain的输入
    runnable_to_execute = RunnableLambda(lambda x: {"input": x.next_inputs}) | chain
    conditional_branches.append((condition, runnable_to_execute))

lcel_router_chain = (
    routing_chain # 步骤 1: 执行路由决策 Chain，输出 Route 对象
    | RunnableBranch(
        # 使用 * 解包操作符，将列表中的所有 (condition, runnable) 元组作为独立参数传入
        *conditional_branches, 
        
        # 默认分支 (作为最后一个参数)
        # 这里的输入 x 是 Route 对象。default chain 只接收 {input} 键。
        # 因此，需要确保 Route 对象的 next_inputs 能够被传递给 default chain 的 {input}
        RunnableLambda(lambda x: {"input": x.next_inputs}) | create_expert_chain("{input}")
    )
)

print("=== LCEL Chain 结构图 ===")
# .get_graph().draw_ascii() 作用：可视化 LCEL 链的结构，对于复杂 Chain 调试非常有帮助。
# 结构图将展示：Routing Chain -> RunnableBranch [Physics Path, Math Path, Default Path]
print(lcel_router_chain.get_graph().draw_ascii())

print("=== LCEL Chain 调用 ===")
# 调用
question_1 = "What is black body radiation?"
print(f"Input: {question_1}")
# 输出将是 physics_chain 的结果
print("LCEL Router Chain 替代 (Physics):", lcel_router_chain.invoke({"input": question_1}))

question_2 = "what is 2 + 2"
print(f"\nInput: {question_2}")
# 输出将是 math_chain 的结果
print("LCEL Router Chain 替代 (Math):", lcel_router_chain.invoke({"input": question_2}))

=== LCEL Chain 结构图 ===
    +-------------+    
    | PromptInput |    
    +-------------+    
           *           
           *           
           *           
+--------------------+ 
| ChatPromptTemplate | 
+--------------------+ 
           *           
           *           
           *           
    +------------+     
    | ChatOpenAI |     
    +------------+     
           *           
           *           
           *           
      +--------+       
      | Lambda |       
      +--------+       
           *           
           *           
           *           
      +--------+       
      | Branch |       
      +--------+       
           *           
           *           
           *           
   +--------------+    
   | BranchOutput |    
   +--------------+    
=== LCEL Chain 调用 ===
Input: What is black body radiation?
LCEL Router Chain 替代 (Physics): Black body radiation refers to the electromagnetic radiation emitted by an idealized object kn

Reminder: Download your notebook to you local computer to save your work.